In [1]:
import os
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import wilcoxon
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold

DATA_FILE = '../data/right.csv'
OUTPUT = '../results/'
os.makedirs(OUTPUT, exist_ok=True)

#taken from Subhash et.al, 2022
FEATURES = ['ax_mean', 'ax_var', 'ay_mean', 'ay_var', 'az_mean', 'az_var',
            'a_norm_mean', 'a_norm_var', 'a_norm_min', 'a_norm_max']

#three CV methods we evaluate
STRATEGIES = ['Random window-level', 'Continuous within-task', 'Leave-one-task-out']

WINDOW = 0.25      # seconds
MIN_SAMPLES = 12   # 0.25 s at 50 Hz
N_FOLDS = 5
N_REPEATS = 10
SEED = 42

In [2]:
def window_features(end_time, window):
    functional = window.gnd.mean()
    return {'time': end_time,
            'subject': window.subject.iloc[0],
            'task': window.task.iloc[0],
            'gnd': np.nan if functional == 0.5 else int(functional > 0.5),
            'ax_mean': window.ax.mean(), 'ax_var': window.ax.var(),
            'ay_mean': window.ay.mean(), 'ay_var': window.ay.var(),
            'az_mean': window.az.mean(), 'az_var': window.az.var(),
            'a_norm_mean': window.a_norm.mean(), 'a_norm_var': window.a_norm.var(),
            'a_norm_min': window.a_norm.min(), 'a_norm_max': window.a_norm.max()}


def load_windows():
    raw = pd.read_csv(DATA_FILE, parse_dates=['time'])
    raw = raw.sort_values(['subject', 'time'])
    raw['task'] = raw.task.fillna('').str.strip()
    raw['a_norm'] = np.sqrt(raw.ax**2 + raw.ay**2 + raw.az**2)

    # new segment at subject change, task change or a gap > 60 s
    gap = raw.time.diff().dt.total_seconds()
    new_segment = (raw.subject != raw.subject.shift()) | (raw.task != raw.task.shift()) | (gap > 60)

    rows = []
    for _, segment in raw.set_index('time').groupby(new_segment.cumsum().values):
        for start, window in segment.resample(str(WINDOW) + 's', origin='start'):
            if len(window) >= MIN_SAMPLES:
                rows.append(window_features(start + pd.Timedelta(seconds=WINDOW), window))

    windows = pd.DataFrame(rows).dropna(subset=['gnd'])
    windows = windows[windows.task != ''].astype({'gnd': int})
    return windows

In [3]:
def shuffled_splits(subject):
    for repeat in range(N_REPEATS):
        folds = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + repeat)
        for train, test in folds.split(subject, subject.gnd):
            yield repeat, train, test


def time_blocks(n):
    blocks = np.array_split(np.arange(n), N_FOLDS)
    return np.concatenate([np.full(len(b), k) for k, b in enumerate(blocks)])


def continuous_splits(subject):
    # segment = consecutive windows of one task with no missing window in between
    gap = subject.time.diff().dt.total_seconds()
    segment = ((subject.task != subject.task.shift()) | (gap > WINDOW)).cumsum()
    block = segment.groupby(segment).transform(lambda s: time_blocks(len(s)))
    for fold in range(N_FOLDS):
        yield 0, np.flatnonzero(block != fold), np.flatnonzero(block == fold)


def task_out_splits(subject):
    for train, test in LeaveOneGroupOut().split(subject, groups=subject.task):
        yield 0, train, test

In [4]:
def run_cv(windows, strategy, splits):
    results = []
    for _, subject in windows.groupby('subject'):
        subject = subject.sort_values('time').reset_index(drop=True)
        for repeat, train, test in splits(subject):
            model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1)
            model.fit(subject.loc[train, FEATURES], subject.gnd[train])
            result = subject.loc[test, ['subject', 'gnd']].assign(strategy=strategy, repeat=repeat)
            result['prediction'] = model.predict(subject.loc[test, FEATURES])
            results.append(result)
    return pd.concat(results)


def scores(result):
    tn, fp, fn, tp = confusion_matrix(result.gnd, result.prediction, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return pd.Series({'sensitivity': sensitivity,
                      'specificity': specificity,
                      'youden_index': sensitivity + specificity - 1})

In [5]:
windows = load_windows()

# exclude subjects with fewer than 5 windows in any task
counts = windows.groupby(['subject', 'task']).size().unstack(fill_value=0)
keep = counts.index[(counts >= N_FOLDS).all(axis=1)]
print('excluded:', list(counts.index.difference(keep)))
windows = windows[windows.subject.isin(keep)]

predictions = pd.concat([run_cv(windows, STRATEGIES[0], shuffled_splits),
                         run_cv(windows, STRATEGIES[1], continuous_splits),
                         run_cv(windows, STRATEGIES[2], task_out_splits)])

excluded: [3, 5, 6]


In [6]:
# Youden index per repeat (10 for shuffled, 1 for others), then averaged per subject
repeat_scores = predictions.groupby(['strategy', 'subject', 'repeat'])[['gnd', 'prediction']].apply(scores)
subject_scores = repeat_scores.groupby(['strategy', 'subject']).mean()

summary = subject_scores.groupby('strategy').agg(['mean', 'std']).loc[STRATEGIES]
summary.round(3).to_csv(OUTPUT + 'performance_summary.csv')
summary.round(3)

sensitivity        specificity        youden_index  \
                              mean    std        mean    std         mean   
strategy                                                                    
Random window-level          0.997  0.002       0.888  0.064        0.885   
Continuous within-task       0.994  0.006       0.873  0.064        0.867   
Leave-one-task-out           0.982  0.024       0.636  0.214        0.618   

                               
                          std  
strategy                       
Random window-level     0.065  
Continuous within-task  0.068  
Leave-one-task-out      0.208

In [7]:
functional = (100 * windows.groupby('subject').gnd.mean()).rename('functional_percent')
by_subject = subject_scores.reset_index().merge(functional, on='subject')
by_subject.round(3).to_csv(OUTPUT + 'performance_by_subject.csv', index=False)
by_subject.round(3)

,strategy,subject,sensitivity,specificity,youden_index,functional_percent
0,Continuous within-task,1,0.999,0.849,0.848,83.367
1,Continuous within-task,2,0.995,0.809,0.803,88.867
2,Continuous within-task,4,0.999,0.946,0.945,80.895
3,Continuous within-task,7,0.999,0.956,0.955,93.639
4,Continuous within-task,8,0.994,0.909,0.903,84.139
5,Continuous within-task,9,0.996,0.840,0.836,92.308
6,Continuous within-task,10,0.981,0.799,0.780,81.250
7,Leave-one-task-out,1,0.975,0.825,0.800,83.367
8,Leave-one-task-out,2,0.986,0.678,0.664,88.867
9,Leave-one-task-out,4,0.999,0.927,0.926,80.895


In [8]:
youden = subject_scores.youden_index.unstack('strategy')

tests = []
for a, b in combinations(STRATEGIES, 2):
    p = wilcoxon(youden[a], youden[b]).pvalue
    tests.append({'comparison': a + ' vs ' + b, 'p': p, 'p_bonferroni': min(3 * p, 1)})

tests = pd.DataFrame(tests)
print(tests.round(3))

                                      comparison      p  p_bonferroni
0  Random window-level vs Continuous within-task  0.297         0.891
1      Random window-level vs Leave-one-task-out  0.016         0.047
2   Continuous within-task vs Leave-one-task-out  0.016         0.047
